In [0]:
%run ../../config/config

In [0]:
spark.sql("DECLARE OR REPLACE VARIABLE bronze_ticks STRING")
spark.sql("DECLARE OR REPLACE VARIABLE silver_ticks STRING")
spark.sql("DECLARE OR REPLACE VARIABLE silver_quarantine STRING")
spark.sql("DECLARE OR REPLACE VARIABLE valid_symbols ARRAY<STRING>")

spark.sql(
    "SET VAR bronze_ticks = :bronze_ticks, silver_ticks = :silver_ticks, silver_quarantine = :silver_quarantine",
    args={"bronze_ticks": BRONZE_EVENTHUB_TICKS, "silver_ticks": SILVER_TICKS, "silver_quarantine": SILVER_QUARANTINE},
)
spark.sql("SET VAR valid_symbols = split(:symbols, ',')", args={"symbols": ",".join(VALID_SYMBOLS)})

In [0]:
%sql
-- 1. Number of records in layers
SELECT 'bronze' AS layer, COUNT(*) AS row_count FROM IDENTIFIER(bronze_ticks)
UNION ALL
SELECT 'silver', COUNT(*) FROM IDENTIFIER(silver_ticks)
UNION ALL
SELECT 'quarantine', COUNT(*) FROM IDENTIFIER(silver_quarantine);

In [0]:
%sql
-- 2. Invalid records in Silver
-- Expected: 0 rows
SELECT *
FROM IDENTIFIER(silver_ticks)
WHERE symbol IS NULL
   OR NOT array_contains(valid_symbols, symbol)
   OR event_time IS NULL
   OR price_usd IS NULL
   OR isnan(price_usd)
   OR price_usd <= 0
   OR price_usd >= CAST('Infinity' AS DOUBLE)
   OR ingestion_timestamp IS NULL
   OR event_time > ingestion_timestamp + INTERVAL 5 MINUTES;

In [0]:
%sql
-- 3. Duplicates
-- Expected: 0 rows
SELECT symbol, event_time, source, COUNT(*) AS copies
FROM IDENTIFIER(silver_ticks)
GROUP BY symbol, event_time, source
HAVING COUNT(*) > 1;

In [0]:
%sql
-- 4. Viewing rejected messages and reasons
SELECT value, failed_rules, quarantine_reason
FROM IDENTIFIER(silver_quarantine);

In [0]:
%sql
-- 5. Rejected records without explanation
-- Expected: 0 rows
SELECT *
FROM IDENTIFIER(silver_quarantine)
WHERE failed_rules IS NULL
   OR size(failed_rules) = 0
   OR quarantine_reason IS NULL
   OR trim(quarantine_reason) = '';